<a href="https://colab.research.google.com/github/frappedegansito/MachineLearning/blob/main/Unidad%203/P4U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
# Importamos librerias necesarias para Árboles de decisión
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [ ]:
prestamos_df = pd.read_csv('prestamos_ok.csv')
prestamos_df.head()

In [ ]:
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']]
# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

In [ ]:
# Dividimos el dataFrame
# stratify es para que mantenga la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y )

In [ ]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
# Crear el modelo de árbol de clasificación
tree_model = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
# Entrenar el modelo
tree_model.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", tree_model.score(X_train, y_train) )

In [ ]:
# Hacer predicciones con los datos de prueba
y_pred = tree_model.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador de Arboles de Decisión: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz Confusion de Arboles de Decisión:\n', confusion_matrix(y_test, y_pred ))


In [ ]:
# Crear modelo con balanceo de clases
tree_model2 = DecisionTreeClassifier(criterion='gini', max_depth=5, class_weight='balanced', random_state=42, )
# Entrenar el modelo
tree_model2.fit(X_train, y_train)
# Realizar fase de pruena
y_pred2 = tree_model2.predict(X_test)
# Evaluar el modelo
print("\nReporte de métricas del clasificador de Arboles de Decisión: \n",
classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))
print("\n Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred2))


In [ ]:
# Crear modelo con balanceo de clases y max_depth (profundidad del árbol)
tree_model3 = DecisionTreeClassifier(criterion='gini', max_depth=7, class_weight='balanced', random_state=42,
# Entrenar el modelo
tree_model3.fit(X_train, y_train)
# Realizar fase de prueba
y_pred3 = tree_model3.predict(X_test)
# Evaluar el modelo de prueba
print("\nReporte de métricas del clasificador de Arboles de Decisión: \n",
classification_report(y_test, y_pred3, target_names=["No Pagado", "Pagado"]))
print("\n Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred3))

In [ ]:
from sklearn.tree import export_text, plot_tree
import matplotlib.pyplot as plt
model_columns = ['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code',
'home_ownership_code', 'annual_inc', 'dti', 'revol_util',
'pub_rec_bankruptcies']
# Visualizar el árbol en texto
print(export_text(tree_model3, feature_names=model_columns))


In [ ]:
# Graficar el árbol
plt.figure(figsize=(20, 10))
plot_tree(tree_model3, feature_names=model_columns, class_names=['Pagado', 'No Pagado'], filled=True)
plt.show()

In [ ]:
importancia_caracteristicas = pd.DataFrame({
'Caracteristica': model_columns,
'Importancia': tree_model3.feature_importances_
}).sort_values(by='Importancia', ascending=False)
print("\nImportancia de las características:")
print(importancia_caracteristicas)

In [ ]:
# Graficar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia_caracteristicas, x="Caracteristica", y="Importancia",
title="Importancia de características",
text_auto=".3f", color="Caracteristica")
fig4.update_layout(width=800, height=600)
fig4.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
'criterion': ['gini', 'entropy'],
'max_depth': [5, 7, 10, None],
'min_samples_split': [2, 5, 10, 20],
'min_samples_leaf': [1, 2, 5, 10],
'class_weight': [None, 'balanced'],
'ccp_alpha': [0.0, 0.001, 0.01]
}
grid_search = GridSearchCV(
DecisionTreeClassifier(random_state=42),
param_grid,
scoring='recall_macro', # o 'f2' si te interesa más los impagos
cv=5,
n_jobs=-1,
verbose=2
)
grid_search.fit(X_train, y_train)
print("Mejores parámetros:", grid_search.best_params_)

In [ ]:
# Evaluación del mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nReporte de Clasificación (modelo optimizado para detectar impagos):\n")
print(classification_report(y_test, y_pred))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
